In [64]:
import os
import re
from dataclasses import dataclass
from typing import Iterable, Optional

import pandas as pd

BASE_DATA_DIR = "/Volumes/default/Zotero/방산/data"


def _version_key(v: str):
    # "6.2" -> (6, 2), "4" -> (4, 0)
    parts = v.split(".")
    major = int(parts[0])
    minor = int(parts[1]) if len(parts) > 1 else 0
    return (major, minor)


def discover_versions(base_dir: str = BASE_DATA_DIR) -> list[str]:
    """Find version folders like r1, r3.1, r6.2 under base_dir."""
    vers: list[str] = []
    for name in os.listdir(base_dir):
        m = re.fullmatch(r"r(\d+(?:\.\d+)?)", name)
        if m and os.path.isdir(os.path.join(base_dir, name)):
            vers.append(m.group(1))
    return sorted(set(vers), key=_version_key)


def read_schema(
    version: str,
    data_name: str,
    base_dir: str = BASE_DATA_DIR,
    *,
    with_dtypes: bool = False,
) -> pd.Series:
    """Return columns (and optionally dtypes) for one version + dataset."""
    path = os.path.join(base_dir, f"r{version}", f"{data_name}.csv")
    if with_dtypes:
        df = pd.read_csv(path, nrows=200)  # sample for dtype inference
        return df.dtypes
    df0 = pd.read_csv(path, nrows=0)
    return pd.Series(index=df0.columns, dtype="object")


@dataclass(frozen=True)
class SchemaDiff:
    from_version: str
    to_version: str
    added: list[str]
    removed: list[str]


def diff_schema(
    from_version: str,
    to_version: str,
    data_name: str,
    base_dir: str = BASE_DATA_DIR,
) -> SchemaDiff:
    a0 = read_schema(from_version, data_name, base_dir, with_dtypes=False)
    b0 = read_schema(to_version, data_name, base_dir, with_dtypes=False)
    a_cols = list(a0.index)
    b_cols = list(b0.index)

    a_set, b_set = set(a_cols), set(b_cols)
    added = sorted(b_set - a_set)
    removed = sorted(a_set - b_set)

    return SchemaDiff(
        from_version=from_version,
        to_version=to_version,
        added=added,
        removed=removed,
    )


def report_schema_changes(
    data_name: str,
    versions: Optional[Iterable[str]] = None,
    base_dir: str = BASE_DATA_DIR,
    *,
    show_all_columns: bool = False,
) -> pd.DataFrame:
    """Compare schemas across versions and return a compact report (add/remove only)."""
    vers = list(versions) if versions is not None else discover_versions(base_dir)
    vers = sorted({str(v) for v in vers}, key=_version_key)

    rows: list[dict] = []

    # Per-version schema (optional)
    if show_all_columns:
        for v in vers:
            try:
                cols = list(read_schema(v, data_name, base_dir, with_dtypes=False).index)
                rows.append(
                    {
                        "from": v,
                        "to": v,
                        "added": "",
                        "removed": "",
                        "dtype_changed": "",
                        "columns": ", ".join(cols),
                    }
                )
            except FileNotFoundError:
                rows.append(
                    {
                        "from": v,
                        "to": v,
                        "added": "",
                        "removed": "",
                        "dtype_changed": "",
                        "columns": "(missing)",
                    }
                )

    # 최초로 파일이 존재하는 버전에서의 컬럼들(베이스라인 컬럼들도 "추가"로 표시)
    base_version: Optional[str] = None
    base_cols: list[str] = []
    for v in vers:
        try:
            base_cols = list(read_schema(v, data_name, base_dir, with_dtypes=False).index)
            base_version = v
            break
        except FileNotFoundError:
            continue
    if base_version is not None:
        rows.append(
            {
                "from": base_version,
                "to": base_version,
                "added": ", ".join(base_cols),
                "removed": "",
            }
        )

    # Diff between consecutive versions
    for a, b in zip(vers, vers[1:]):
        try:
            d = diff_schema(a, b, data_name, base_dir)
            rows.append(
                {
                    "from": a,
                    "to": b,
                    "added": ", ".join(d.added),
                    "removed": ", ".join(d.removed),
                }
            )
        except FileNotFoundError:
                rows.append(
                    {
                        "from": a,
                        "to": b,
                        "added": "(missing file)",
                        "removed": "(missing file)",
                    }
                )

    df = pd.DataFrame(rows)
    return df


def combine_schema_reports(
    data_names: Iterable[str],
    versions: Optional[Iterable[str]] = None,
    base_dir: str = BASE_DATA_DIR,
    *,
    only_changes: bool = True,
) -> pd.DataFrame:
    """Build ONE table by stacking multiple datasets' schema-diff reports."""
    vers = list(versions) if versions is not None else discover_versions(base_dir)

    frames: list[pd.DataFrame] = []
    for name in data_names:
        df = report_schema_changes(
            name,
            versions=vers,
            base_dir=base_dir,
            show_all_columns=False,
        ).copy()
        df.insert(0, "dataset", name)
        frames.append(df)

    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    if only_changes and not out.empty:
        def _is_change(s: pd.Series) -> pd.Series:
            return (
                s.fillna("").astype(str).str.strip().ne("")
                & s.fillna("").astype(str).str.strip().ne("(missing file)")
            )

        mask = _is_change(out["added"]) | _is_change(out["removed"])
        out = out.loc[mask].reset_index(drop=True)

    return out


_SIMPLE_DTYPE_MAP = {
    "int64": "int",
    "Int64": "int",
    "float64": "float",
    "Float64": "float",
    "object": "str",
    "string": "str",
    "bool": "bool",
    "boolean": "bool",
    "datetime64[ns]": "datetime",
    "category": "category",
}


def _simplify_dtype(dt: str) -> str:
    return _SIMPLE_DTYPE_MAP.get(dt, dt)


def summarize_dtypes(
    data_name: str,
    versions: Optional[Iterable[str]] = None,
    base_dir: str = BASE_DATA_DIR,
) -> pd.DataFrame:
    """Return which simple dtypes (int/float/str/...) appear in each version."""
    vers = list(versions) if versions is not None else discover_versions(base_dir)
    vers = sorted({str(v) for v in vers}, key=_version_key)

    rows: list[dict] = []
    for v in vers:
        try:
            dtypes = read_schema(v, data_name, base_dir, with_dtypes=True)
            used = sorted({_simplify_dtype(str(t)) for t in dtypes})
            rows.append(
                {
                    "dataset": data_name,
                    "version": v,
                    "dtypes": ", ".join(used),
                }
            )
        except FileNotFoundError:
            rows.append(
                {
                    "dataset": data_name,
                    "version": v,
                    "dtypes": "(missing file)",
                }
            )

    return pd.DataFrame(rows)


def summarize_dtypes_all(
    data_names: Iterable[str],
    versions: Optional[Iterable[str]] = None,
    base_dir: str = BASE_DATA_DIR,
) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for name in data_names:
        frames.append(summarize_dtypes(name, versions=versions, base_dir=base_dir))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _format_version_span(versions: list[str]) -> str:
    if len(versions) == 1:
        return versions[0]
    return f"{versions[0]}–{versions[-1]}"


def compact_dtype_summary(
    long_df: pd.DataFrame,
    *,
    drop_missing: bool = True,
) -> pd.DataFrame:
    """Consecutive versions with same dtypes -> one row; drop missing files."""
    if long_df.empty:
        return long_df
    df = long_df.copy()
    if drop_missing:
        df = df[df["dtypes"].astype(str).str.strip() != "(missing file)"].reset_index(
            drop=True
        )
    if df.empty:
        return pd.DataFrame(columns=["dataset", "versions", "dtypes"])

    rows: list[dict] = []
    for dataset in df["dataset"].unique():
        sub = df[df["dataset"] == dataset].copy()
        sub["_vk"] = sub["version"].astype(str).map(_version_key)
        sub = sub.sort_values("_vk")

        current: Optional[str] = None
        chunk: list[str] = []
        for _, r in sub.iterrows():
            d = str(r["dtypes"]).strip()
            v = str(r["version"])
            if current is None:
                current = d
                chunk = [v]
            elif d == current:
                chunk.append(v)
            else:
                rows.append(
                    {
                        "dataset": dataset,
                        "versions": _format_version_span(chunk),
                        "dtypes": current,
                    }
                )
                current = d
                chunk = [v]
        if chunk and current is not None:
            rows.append(
                {
                    "dataset": dataset,
                    "versions": _format_version_span(chunk),
                    "dtypes": current,
                }
            )

    return pd.DataFrame(rows)


def summarize_dtypes_compact(
    data_names: Iterable[str],
    versions: Optional[Iterable[str]] = None,
    base_dir: str = BASE_DATA_DIR,
) -> pd.DataFrame:
    long_df = summarize_dtypes_all(data_names, versions=versions, base_dir=base_dir)
    return compact_dtype_summary(long_df)


def read_ldap_csv_schema(
    version: str,
    *,
    csv_name: str = "2009-12.csv",
    base_dir: str = BASE_DATA_DIR,
    nrows: int = 500,
) -> pd.DataFrame:
    """LDAP/{csv_name} 한 파일의 컬럼명 + 단순 dtype 표."""
    path = os.path.join(base_dir, f"r{version}", "LDAP", csv_name)
    if not os.path.isfile(path):
        return pd.DataFrame()
    df = pd.read_csv(path, nrows=nrows)
    return pd.DataFrame(
        [
            {"column": c, "dtype": _simplify_dtype(str(df[c].dtype))}
            for c in df.columns
        ]
    )


def ldap_csv_schema_long(
    *,
    csv_name: str = "2009-12.csv",
    versions: Optional[Iterable[str]] = None,
    base_dir: str = BASE_DATA_DIR,
    nrows: int = 500,
) -> pd.DataFrame:
    """모든 버전: version | column | dtype (긴 표)."""
    vers = list(versions) if versions is not None else discover_versions(base_dir)
    vers = sorted({str(v) for v in vers}, key=_version_key)
    parts: list[pd.DataFrame] = []
    for v in vers:
        sub = read_ldap_csv_schema(
            v, csv_name=csv_name, base_dir=base_dir, nrows=nrows
        )
        if sub.empty:
            parts.append(
                pd.DataFrame(
                    [{"version": v, "column": "(no file)", "dtype": ""}]
                )
            )
        else:
            parts.append(sub.assign(version=v))
    out = pd.concat(parts, ignore_index=True)
    return out[["version", "column", "dtype"]]


def ldap_csv_schema_wide(
    *,
    csv_name: str = "2009-12.csv",
    versions: Optional[Iterable[str]] = None,
    base_dir: str = BASE_DATA_DIR,
    nrows: int = 500,
    missing_as: str = "no",
) -> pd.DataFrame:
    """컬럼명=행, 버전=열, dtype. 해당 버전에 컬럼 없으면 missing_as(기본 'no')."""
    long_df = ldap_csv_schema_long(
        csv_name=csv_name,
        versions=versions,
        base_dir=base_dir,
        nrows=nrows,
    )
    long_df = long_df[long_df["column"] != "(no file)"].copy()
    if long_df.empty:
        return pd.DataFrame()
    wide = long_df.pivot_table(
        index="column", columns="version", values="dtype", aggfunc="first"
    )
    if missing_as:
        wide = wide.fillna(missing_as)
    return wide


def to_latex_table(
    df: pd.DataFrame,
    *,
    caption: Optional[str] = None,
    label: Optional[str] = None,
    longtable: bool = True,
) -> str:
    """Return LaTeX code (booktabs) suitable for papers."""
    if df.empty:
        return "% (empty table)"

    latex = df.to_latex(
        index=False,
        escape=True,
        longtable=longtable,
        caption=caption,
        label=label,
        bold_rows=False,
        multicolumn=True,
        multicolumn_format="c",
        na_rep="",
        column_format=None,
        float_format=None,
        decimal=".",
        sparsify=False,
        formatters=None,
    )

    # Ensure booktabs rules even if pandas defaults change
    if "\\toprule" not in latex:
        latex = latex.replace("\\hline", "\\toprule", 1)
        latex = latex.replace("\\hline", "\\midrule", 1)
        latex = latex[::-1].replace("enilh\\"[::-1], "\\bottomrule"[::-1], 1)[::-1]

    return latex


def pretty_display(df: pd.DataFrame):
    """Nice notebook rendering (wrap long text, left-align).

    Falls back to plain DataFrame if Styler/jinja2 is not available.
    """
    if df.empty:
        return df
    try:
        return (
            df.style
            .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
            .set_table_styles([
                {"selector": "th", "props": [("text-align", "left")]},
                {"selector": "td", "props": [("text-align", "left")]},
            ])
        )
    except Exception:
        # e.g. AttributeError: The '.style' accessor requires jinja2
        return df

In [65]:
# 사용 예시: 각 데이터셋에서 어떤 dtype이 쓰이는지 요약

versions = discover_versions()
data_names = ["device", "logon", "email", "http", "psychometric", "file", "decoy_file"]

# 긴 표 대신: 같은 dtype이 연속인 버전은 한 줄 (예: device 1–6.2 | str, email 2 | str, 3.1–6.2 | int, str)
dtype_compact = summarize_dtypes_compact(data_names, versions=versions)
display(pretty_display(dtype_compact))

print(
    to_latex_table(
        dtype_compact,
        caption="Dtypes by dataset and version span",
        label="tab:schema-dtypes",
        longtable=False,
    )
)


,dataset,versions,dtypes
0,device,1–6.2,str
1,logon,1–6.2,str
2,email,2,str
3,email,3.1–6.2,"int, str"
4,http,1–6.2,str
5,psychometric,2–6.2,"int, str"
6,file,3.1–4.2,str
7,file,5.1–6.2,"bool, str"
8,decoy_file,5.1–6.2,str


\begin{table}
\caption{Dtypes by dataset and version span}
\label{tab:schema-dtypes}
\begin{tabular}{lll}
\toprule
dataset & versions & dtypes \\
\midrule
device & 1–6.2 & str \\
logon & 1–6.2 & str \\
email & 2 & str \\
email & 3.1–6.2 & int, str \\
http & 1–6.2 & str \\
psychometric & 2–6.2 & int, str \\
file & 3.1–4.2 & str \\
file & 5.1–6.2 & bool, str \\
decoy\_file & 5.1–6.2 & str \\
\bottomrule
\end{tabular}
\end{table}



In [66]:
# 사용 예시: 버전 자동 탐색 후, 여러 표를 합쳐서 한 번에 출력 + 논문용 LaTeX

versions = discover_versions()
print("discovered versions:", versions)

data_names = ["device", "logon", "email", "http", "psychometric", "file", "decoy_file"]

# 1) 합친 단일 표 (변경 있는 행만)
combined = combine_schema_reports(data_names, versions=versions, only_changes=True)
display(combined)

# 2) LaTeX로 뽑기 (그대로 논문에 붙여넣기)
latex = to_latex_table(
    combined,
    caption="Schema changes across dataset versions",
    label="tab:schema-changes",
    longtable=True,
)
print(latex)

# dtype까지 비교하고 싶으면(check_dtypes=True):
# combined_types = combine_schema_reports(data_names, versions=versions, check_dtypes=True, only_changes=True)
# display(pretty_display(combined_types))
# print(to_latex_table(combined_types, caption="Schema + dtype changes", label="tab:schema-dtype-changes"))


discovered versions: ['1', '2', '3.1', '3.2', '4.1', '4.2', '5.1', '5.2', '6.1', '6.2']


,dataset,from,to,added,removed
0,device,1,1,"id, date, user, pc, activity",
1,device,4.2,5.1,file_tree,
2,logon,1,1,"id, date, user, pc, activity",
3,email,2,2,"id, date, to, from",
4,email,2,3.1,"attachments, content, size",
5,email,3.2,4.1,"bcc, cc, pc, user",
6,email,4.2,5.1,activity,
7,http,1,1,"{M8H9-W9NL75TH-1322KOLO}, 01/04/2010 07:08:47,...",
8,http,1,2,"date, id, pc, url, user","01/04/2010 07:08:47, DTAA/AMA0606, PC-1514, ht..."
9,http,2,3.1,content,


\begin{longtable}{lllll}
\caption{Schema changes across dataset versions} \label{tab:schema-changes} \\
\toprule
dataset & from & to & added & removed \\
\midrule
\endfirsthead
\caption[]{Schema changes across dataset versions} \\
\toprule
dataset & from & to & added & removed \\
\midrule
\endhead
\midrule
\multicolumn{5}{r}{Continued on next page} \\
\midrule
\endfoot
\bottomrule
\endlastfoot
device & 1 & 1 & id, date, user, pc, activity &  \\
device & 4.2 & 5.1 & file\_tree &  \\
logon & 1 & 1 & id, date, user, pc, activity &  \\
email & 2 & 2 & id, date, to, from &  \\
email & 2 & 3.1 & attachments, content, size &  \\
email & 3.2 & 4.1 & bcc, cc, pc, user &  \\
email & 4.2 & 5.1 & activity &  \\
http & 1 & 1 & \{M8H9-W9NL75TH-1322KOLO\}, 01/04/2010 07:08:47, DTAA/AMA0606, PC-1514, http://cnet.com &  \\
http & 1 & 2 & date, id, pc, url, user & 01/04/2010 07:08:47, DTAA/AMA0606, PC-1514, http://cnet.com, \{M8H9-W9NL75TH-1322KOLO\} \\
http & 2 & 3.1 & content &  \\
http & 5.2 & 6.1 & 

In [60]:
# LDAP/2009-12.csv — 버전별 컬럼 & 단순 dtype (str, int, …)

versions = discover_versions()

# 한 표에 모두: version | column | dtype
ldap_200912 = ldap_csv_schema_long(csv_name="2009-12.csv", versions=versions)
display(pretty_display(ldap_200912))

# 버전마다 작은 표로 보기
for v in versions:
    sub = read_ldap_csv_schema(v, csv_name="2009-12.csv")
    if sub.empty:
        print(f"r{v}/LDAP/2009-12.csv — 없음")
    else:
        print(f"\n=== r{v} LDAP/2009-12.csv ===")
        display(pretty_display(sub.assign(version=v).reindex(columns=["version", "column", "dtype"])))

# 컬럼 × 버전 dtype 비교 (논문용)
# wide: 없는 컬럼은 'no' (해당 버전에 필드 없음). 한글로 쓰려면 missing_as="없음"
ldap_wide = ldap_csv_schema_wide(
    csv_name="2009-12.csv", versions=versions, missing_as="no"
)
ldap_wide_tbl = ldap_wide.reset_index()
ldap_wide_tbl.columns.name = None
display(pretty_display(ldap_wide_tbl))

print(
    to_latex_table(
        ldap_wide_tbl,
        caption="LDAP 2009-12.csv: column dtypes by release (no = column absent)",
        label="tab:ldap-200912-wide",
        longtable=True,
    )
)

,version,column,dtype
0,1,employee_name,str
1,1,user_id,str
2,1,Domain,str
3,1,Email,str
4,1,Role,str
5,2,employee_name,str
6,2,user_id,str
7,2,email,str
8,2,role,str
9,3.1,employee_name,str



=== r1 LDAP/2009-12.csv ===


,version,column,dtype
0,1,employee_name,str
1,1,user_id,str
2,1,Domain,str
3,1,Email,str
4,1,Role,str



=== r2 LDAP/2009-12.csv ===


,version,column,dtype
0,2,employee_name,str
1,2,user_id,str
2,2,email,str
3,2,role,str


...

,version,column,dtype
0,6.1,employee_name,str
1,6.1,user_id,str
2,6.1,email,str
3,6.1,role,str
4,6.1,projects,str
5,6.1,business_unit,int
6,6.1,functional_unit,str
7,6.1,department,str
8,6.1,team,str
9,6.1,supervisor,str



=== r6.2 LDAP/2009-12.csv ===


,version,column,dtype
0,6.2,employee_name,str
1,6.2,user_id,str
2,6.2,email,str
3,6.2,role,str
4,6.2,projects,str
5,6.2,business_unit,int
6,6.2,functional_unit,str
7,6.2,department,str
8,6.2,team,str
9,6.2,supervisor,str


,column,1,2,3.1,3.2,4.1,4.2,5.1,5.2,6.1,6.2
0,Domain,str,no,no,no,no,no,no,no,no,no
1,Email,str,no,no,no,no,no,no,no,no,no
2,Role,str,no,no,no,no,no,no,no,no,no
3,business_unit,no,no,int,int,int,int,str,str,int,int
4,department,no,no,str,str,str,str,str,str,str,str
5,email,no,str,str,str,str,str,str,str,str,str
6,employee_name,str,str,str,str,str,str,str,str,str,str
7,functional_unit,no,no,str,str,str,str,str,str,str,str
8,projects,no,no,no,no,no,no,str,str,str,str
9,role,no,str,str,str,str,str,str,str,str,str


\begin{longtable}{lllllllllll}
\caption{LDAP 2009-12.csv: column dtypes by release (no = column absent)} \label{tab:ldap-200912-wide} \\
\toprule
column & 1 & 2 & 3.1 & 3.2 & 4.1 & 4.2 & 5.1 & 5.2 & 6.1 & 6.2 \\
\midrule
\endfirsthead
\caption[]{LDAP 2009-12.csv: column dtypes by release (no = column absent)} \\
\toprule
column & 1 & 2 & 3.1 & 3.2 & 4.1 & 4.2 & 5.1 & 5.2 & 6.1 & 6.2 \\
\midrule
\endhead
\midrule
\multicolumn{11}{r}{Continued on next page} \\
\midrule
\endfoot
\bottomrule
\endlastfoot
Domain & str & no & no & no & no & no & no & no & no & no \\
Email & str & no & no & no & no & no & no & no & no & no \\
Role & str & no & no & no & no & no & no & no & no & no \\
business\_unit & no & no & int & int & int & int & str & str & int & int \\
department & no & no & str & str & str & str & str & str & str & str \\
email & no & str & str & str & str & str & str & str & str & str \\
employee\_name & str & str & str & str & str & str & str & str & str & str \\
functional\_unit & n